# Addendum: Real-vs-Real Baseline for the GAN Similarity Metrics

**Why this exists:** the synthetic-vs-real cosine similarity (0.956) and Frechet Encoder
Distance (8.63) from `gan-mel-synthesis.ipynb` are uninterpretable on their own -- there is
no reference point for what "high" or "low" means in this embedding space for dermoscopy
images. This notebook computes the same two metrics **between two random halves of the real
MEL images themselves**, giving a same-domain baseline to compare against.

Reads only the existing `expA_train.csv` and `ssl_encoder_real_24000.pth` -- no new
training, runs in well under a minute.


In [25]:
import sys
import os
import glob
import random
import importlib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torchvision import transforms
from PIL import Image
from scipy.linalg import sqrtm

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Reproducibility
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

# ==========================================
# 1. DYNAMIC INPUT PATH RESOLUTION
# ==========================================

# A. Locate and load config.py dynamically
config_files = glob.glob('/kaggle/input/**/config.py', recursive=True)
if config_files:
    config_dir = os.path.dirname(config_files[0])
    if config_dir not in sys.path:
        sys.path.insert(0, config_dir)
    import config
    importlib.reload(config)
    print(f"Loaded config from: {config_files[0]}")
else:
    print("Warning: config.py not found in /kaggle/input/. Proceeding with defaults.")

# B. Locate encoder weights (.pt or .pth) dynamically
weight_files = glob.glob('/kaggle/input/**/ssl_encoder_real_24000.*', recursive=True)
if not weight_files:
    raise FileNotFoundError("Could not find ssl_encoder_real_24000 weights file in /kaggle/input/")
ENCODER_PATH = weight_files[0]
print(f"Found encoder weights at: {ENCODER_PATH}")

# D. Locate ISIC image directory dynamically
image_files = glob.glob('/kaggle/input/**/ISIC_*.jpg', recursive=True)

if image_files:
    LABELED_DIR = os.path.dirname(image_files[0])
else:
    # Fallback search for any jpg folder inside ISIC datasets
    all_jpgs = glob.glob('/kaggle/input/*ISIC*/**/*.jpg', recursive=True)
    if all_jpgs:
        LABELED_DIR = os.path.dirname(all_jpgs[0])
    else:
        raise FileNotFoundError("Could not locate ISIC image files in /kaggle/input/")

print(f"Using image directory: {LABELED_DIR}")

# E. Set writable output directory
OUTPUT_DIR = '/kaggle/working/outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ==========================================
# 2. MODEL ARCHITECTURE & LOAD WEIGHTS
# ==========================================

class SimpleEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten()
        )

    def forward(self, x):
        return self.features(x)

encoder = SimpleEncoder().to(device)
encoder.load_state_dict(torch.load(ENCODER_PATH, map_location=device))
encoder.eval()
print("Encoder loaded successfully!")

# ==========================================
# 3. HELPER & METRIC FUNCTIONS
# ==========================================

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

def embed_images(paths):
    feats = []
    with torch.no_grad():
        for p in paths:
            img = Image.open(p).convert('RGB')
            x = eval_transform(img).unsqueeze(0).to(device)
            feats.append(encoder(x).cpu().numpy().squeeze(0))
    return np.stack(feats)

def cosine_sim_matrix(a, b):
    a_n = a / (np.linalg.norm(a, axis=1, keepdims=True) + 1e-8)
    b_n = b / (np.linalg.norm(b, axis=1, keepdims=True) + 1e-8)
    return a_n @ b_n.T

def frechet_distance(feats_a, feats_b):
    mu_a, mu_b = feats_a.mean(0), feats_b.mean(0)
    cov_a, cov_b = np.cov(feats_a, rowvar=False), np.cov(feats_b, rowvar=False)
    
    diff = mu_a - mu_b
    covmean, _ = sqrtm(cov_a.dot(cov_b), disp=False)
    
    # Handle imaginary numbers caused by float precision limits
    if np.iscomplexobj(covmean):
        covmean = covmean.real

    tr_covmean = np.trace(covmean)
    return diff.dot(diff) + np.trace(cov_a) + np.trace(cov_b) - 2 * tr_covmean

# ==========================================
# 4. DATA PROCESSING & METRIC COMPUTATION
# ==========================================

train_df = pd.read_csv(CSV_PATH)
mel_df = train_df[train_df['label'] == 'MEL'].sample(frac=1, random_state=42).reset_index(drop=True)

half = len(mel_df) // 2
group_a = mel_df.iloc[:half]
group_b = mel_df.iloc[half:]
print(f"Real MEL split baseline: {len(group_a)} vs {len(group_b)} images")

paths_a = [os.path.join(LABELED_DIR, f"{r.image}.jpg") for _, r in group_a.iterrows()]
paths_b = [os.path.join(LABELED_DIR, f"{r.image}.jpg") for _, r in group_b.iterrows()]

feats_a = embed_images(paths_a)
feats_b = embed_images(paths_b)

sim = cosine_sim_matrix(feats_a, feats_b)
fd = frechet_distance(feats_a, feats_b)

print(f"Mean Cosine Similarity: {sim.mean():.4f}")
print(f"Fréchet Distance: {fd:.4f}")

Loaded config from: /kaggle/input/datasets/ittisamurtunib/skin-ssl-code/skin-ssl-code/config.py
Found encoder weights at: /kaggle/input/datasets/ittisamurtunib/ssl-encoder-real-24000/ssl_encoder_real_24000.pth
Using image directory: /kaggle/input/datasets/ittisamurtunib/dataset/ISIC_2019_Training_Input/ISIC_2019_Training_Input
Encoder loaded successfully!
Real MEL split baseline: 17 vs 18 images
Mean Cosine Similarity: 0.8003
Fréchet Distance: 2.1301


/tmp/ipykernel_58/1842306215.py:125: DeprecationWarning: The `disp` argument is deprecated and will be removed in SciPy 1.18.0.
  covmean, _ = sqrtm(cov_a.dot(cov_b), disp=False)
